# Sign Language Translation — Colab / H100 Training

**Translate sign-language video / pose-keypoint sequences into spoken text** via a gloss intermediate (Sign2Gloss2Text). A frozen pose front-end (MediaPipe Holistic on Colab, a SeedPoseEngine offline) feeds a **trainable seq2seq core** (a transformer pose→gloss recognizer + a t5-small gloss→text translator). A deterministic agent segments the signs, gates recognition confidence, verifies the translation, and **abstains** on out-of-vocabulary signing.

**One button:** run the *Setup* cells, then **ONE-BUTTON AUTOPILOT**. It auto-detects the GPU (H100 → A100 → L4 → T4), trains the core, evaluates vs baselines, runs analysis, and writes `report.pdf` + `slides.pptx` + a submission bundle to your Drive.

_Author: Le Dinh Minh Quan (23127460) — NLP in Industry, Final Assignment (P01, the last project)._

> Note: every continuous sign-language corpus is non-commercial / gated (see `docs/data_card.md`), so the **synthetic pose generator is the primary data** — a defensible, honest design choice.

## 0. Controls

In [ ]:
#@title Controls { run: 'auto' }
USE_DRIVE = True           #@param {type:'boolean'}
CLONE_FROM_GIT = True      #@param {type:'boolean'}
GIT_URL = 'https://github.com/<your-username>/01_Sign_Language_Translation.git'  #@param {type:'string'}
TRAIN_CORE = True          #@param {type:'boolean'}
TRAIN_TRANSLATOR = True    #@param {type:'boolean'}   # fine-tune t5 gloss->text (else lexicon)
N_SENTENCES = 1200         #@param {type:'integer'}
RUN_AUTOPILOT = True       #@param {type:'boolean'}
print('controls set')

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo 'No GPU — runtime > Change runtime type > GPU (H100/A100/L4/T4)'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Drive mount + paths

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/signlang'
else:
    BASE = '/content/signlang'
os.makedirs(BASE, exist_ok=True)
os.environ['SIGNLANG_ARTIFACTS_DIR'] = BASE + '/artifacts'
os.environ['HF_HOME'] = BASE + '/hf'
print('artifacts ->', os.environ['SIGNLANG_ARTIFACTS_DIR'])

## 3. Get the code

In [ ]:
import os
if CLONE_FROM_GIT:
    if not os.path.isdir('/content/repo'):
        !git clone $GIT_URL /content/repo
    else:
        !cd /content/repo && git pull --ff-only || true
    PROJ = '/content/repo'
else:
    PROJ = BASE + '/01_Sign_Language_Translation'
os.environ['PROJ'] = PROJ
assert os.path.isdir(PROJ + '/src/signlang'), 'repo not found at ' + PROJ
print('repo at', PROJ)

## 4. Install (Colab-safe)
Install the ML/serving/report deps from `requirements_colab.txt` (torch is preinstalled on Colab), then the package with `--no-deps` so it does not perturb Colab's resolved torch/CUDA.

In [ ]:
!pip -q install -r $PROJ/requirements_colab.txt
!pip -q install -e $PROJ --no-deps
import importlib, signlang; importlib.reload(signlang)
print('signlang', signlang.__version__)

## 5. GPU auto-profile → write `train_colab.yaml`
Batch size auto-scales by GPU tier (H100 → A100 → L4 → T4).

In [ ]:
import torch, yaml, os
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
if 'H100' in name: bs, bf16, tf32 = 192, True, True
elif 'A100' in name: bs, bf16, tf32 = 128, True, True
elif 'L4' in name:  bs, bf16, tf32 = 64, True, True
elif 'T4' in name:  bs, bf16, tf32 = 32, False, False
else:               bs, bf16, tf32 = 16, False, False
cfg = {'data': {'use_hf': False, 'n_sentences': int(N_SENTENCES), 'vocab_size': 40, 'seed': 42},
       'pose': {'engine': 'auto', 'normalize': True},
       'model': {'arch': 'pose2seq_transformer', 'd_model': 256, 'enc_layers': 4,
                 'text_backbone': 't5-small', 'use_gloss_intermediate': bool(TRAIN_TRANSLATOR),
                 'num_train_epochs': 25, 'per_device_train_batch_size': bs, 'bf16': bf16, 'tf32': tf32},
       'agent': {'llm_fallback_enabled': False}}
os.makedirs(PROJ + '/configs', exist_ok=True)
open(PROJ + '/configs/train_colab.yaml','w').write(yaml.safe_dump(cfg, sort_keys=False))
print(f'GPU={name} -> batch_size={bs} bf16={bf16} tf32={tf32}')
print(open(PROJ + '/configs/train_colab.yaml').read())

## 6. Sanity-check + render a synthetic collection

In [ ]:
!cd $PROJ && signlang --config configs/train_colab.yaml data
!cd $PROJ && signlang --config configs/train_colab.yaml gen-synthetic

## 7. ONE-BUTTON AUTOPILOT 🚀
data → baseline → **train recognizer (+ t5 translator)** → evaluate → tune → error-analysis → quality → benchmark → demo → monitoring → **report.pdf + slides.pptx** → grade → zipped bundle. Each step is isolated; training is skipped automatically if no GPU is present.

In [ ]:
flag = '' if TRAIN_CORE else '--no-train'
if RUN_AUTOPILOT:
    !cd $PROJ && signlang --config configs/train_colab.yaml autopilot $flag
else:
    print('RUN_AUTOPILOT is off — use the individual steps below.')

## 8. Individual steps (optional)

In [ ]:
# Train only (recognizer + t5 translator):
# !cd $PROJ && signlang --config configs/train_colab.yaml train
# Evaluate (full, with the trained core):
# !cd $PROJ && signlang --config configs/train_colab.yaml evaluate
# Pose-noise robustness sweep:
# !cd $PROJ && signlang --config configs/train_colab.yaml tune
# Report + slides only:
# !cd $PROJ && signlang --config configs/train_colab.yaml generate-report
# !cd $PROJ && signlang --config configs/train_colab.yaml generate-slides

## 9. Diagnostics

In [ ]:
!cd $PROJ && signlang --config configs/train_colab.yaml grade
!cd $PROJ && signlang --config configs/train_colab.yaml demo-agent

## 10. Test the trained model

In [ ]:
from signlang.config import load_config
from signlang.agent.translate_agent import TranslationAgent
from signlang.data.synth_pose import make_sentence
cfg = load_config(PROJ + '/configs/train_colab.yaml')
agent = TranslationAgent(cfg, load_model=True)  # loads the trained recognizer/translator if present
for s in [5000, 5001, 5002, 7777]:
    seq = make_sentence(s, cfg)
    out = agent.translate(seq)
    print(s, '->', out['glosses'], '=>', repr(out['text']), '| gold:', repr(seq.spec['text']),
          '| abstained=', out['abstained'])

## 11. Locate deliverables

In [ ]:
import glob, os
root = os.environ['SIGNLANG_ARTIFACTS_DIR']
for pat in ['submission/*/report.pdf','submission/*/slides.pptx','submission/*/submission_bundle.zip',
            'runs/*/eval.json','models/*']:
    for p in glob.glob(os.path.join(root, pat)):
        print(p)
print('\nDownload report.pdf + slides.pptx + submission_bundle.zip from the path above (in your Drive).')